# Entraînement final — SVM

Ce notebook :

1. lit automatiquement la meilleure configuration produite par `hyperparameter_search.py`,
2. réentraîne le pipeline **StandardScaler + PCA + SVM** sur **l'intégralité du pool d'entraînement** (train+validation concaténés),
3. sauvegarde le pipeline final dans `results/svm/final_model/`.

Le jeu de test final n'est **pas** touché ici — il sera utilisé uniquement dans `evaluate.ipynb`.

In [6]:
import gc, json, sys, time
from datetime import datetime
from pathlib import Path

import numpy as np
import joblib
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while not (ROOT / "cross_validation.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ipynb.fs.full.preprocessing import get_data_pipeline

FINAL_DIR = ROOT / "results" / "svm" / "final_model"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = ROOT / "results" / "svm" / "hyperparameter_search" / "best_config.json"

# Doit correspondre à la valeur utilisée dans hyperparameter_search.py.
AUGMENT = True
SEED = 42

## 1. Lecture de la meilleure configuration

La configuration provient directement du fichier JSON produit par la recherche d'hyperparamètres.

In [7]:
with open(BEST_PATH) as f:
    best = json.load(f)
cfg = best["best_config"]
print("Meilleure configuration :")
print(json.dumps(cfg, indent=2))
print("\nMétriques CV associées :")
print(json.dumps(best["best_metrics"], indent=2))

Meilleure configuration :
{
  "C": 10,
  "gamma": "scale",
  "kernel": "rbf",
  "pca_variance": 0.95
}

Métriques CV associées :
{
  "accuracy_mean": 0.7543509558434932,
  "precision_mean": 0.7410288055842753,
  "recall_mean": 0.7401126746043133,
  "f1_mean": 0.7373414374916075
}


## 2. Extraction des features (pool complet)

On entraîne sur **tout** le pool train+validation. Pas de split de validation interne ici : la sélection des hyperparamètres a déjà été faite par CV.

In [8]:
# Normalisation ImageNet appliquée par le pipeline : on l'inverse avant
# la conversion en niveaux de gris.
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

def extract_features(dataset, batch_size=32, num_workers=0, desc="Extracting"):
    """Niveaux de gris aplatis (224*224=50176) + labels depuis un dataset.

    Le SVM opère sur des vecteurs plats : on dénormalise (ImageNet -> [0, 1]),
    on convertit en niveaux de gris (radiographies monochromes), on aplatit.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers)
    feats, labels = [], []
    for batch in tqdm(loader, desc=desc):
        images = batch["image"].numpy()
        images = np.clip(images * IMAGENET_STD + IMAGENET_MEAN, 0, 1)
        gray = (0.2989 * images[:, 0] + 0.5870 * images[:, 1]
                + 0.1140 * images[:, 2])
        feats.append(gray.reshape(gray.shape[0], -1).astype(np.float32))
        labels.append(batch["label"].numpy())
    return np.concatenate(feats), np.concatenate(labels)

In [9]:
pipeline_data = get_data_pipeline(augment=AUGMENT)
train_view = pipeline_data["train_pool_train_view"]
print(f"Pool d'entraînement final : {len(train_view)} images (augment={AUGMENT})")

X_train, y_train = extract_features(train_view, desc="Extraction train pool")
print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")

Pool d'entraînement final : 5227 images (augment=True)


Extraction train pool: 100%|██████████| 164/164 [01:32<00:00,  1.76it/s]


X_train: (5227, 50176)  |  y_train: (5227,)


## 3. Entraînement du pipeline scaler + PCA + SVM

- **StandardScaler** : centre/réduit les pixels,
- **PCA** : réduit la dimension en conservant `pca_variance` de la variance,
- **SVM** : classifie les features réduites. `probability=True` active les
  estimations de probabilité, nécessaires au ROC-AUC dans `evaluate.ipynb`.

In [10]:
t0 = time.time()

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)

pca = PCA(n_components=cfg["pca_variance"], random_state=SEED)
X_train_p = pca.fit_transform(X_train_s)
print(f"PCA : {X_train.shape[1]} -> {pca.n_components_} composantes "
      f"({pca.explained_variance_ratio_.sum():.4f} variance)")

svm = SVC(kernel=cfg["kernel"], C=cfg["C"], gamma=cfg["gamma"],
          probability=True, random_state=SEED)
svm.fit(X_train_p, y_train)

elapsed = time.time() - t0
train_acc = accuracy_score(y_train, svm.predict(X_train_p))
print(f"\nEntraînement terminé en {elapsed:.1f}s")
print(f"Train accuracy : {train_acc:.4f}")
print(f"Vecteurs de support : {int(svm.support_vectors_.shape[0])}")

PCA : 50176 -> 968 composantes (0.9500 variance)

Entraînement terminé en 193.1s
Train accuracy : 0.9922
Vecteurs de support : 4295


## 4. Sauvegarde du modèle final

Pipeline complet (scaler + PCA + SVM) + métadonnées sauvegardés dans `results/svm/final_model/`.

In [11]:
pipeline_path = FINAL_DIR / "svm_pipeline.joblib"
joblib.dump({"scaler": scaler, "pca": pca, "svm": svm}, pipeline_path)

meta = {
    "model": "SVM",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "config": cfg,
    "augment": AUGMENT,
    "train_pool_size": int(len(train_view)),
    "pca_components": int(pca.n_components_),
    "training_time_s": round(float(elapsed), 1),
    "train_accuracy": float(train_acc),
    "total_support_vectors": int(svm.support_vectors_.shape[0]),
    "pipeline_file": pipeline_path.name,
}
with open(FINAL_DIR / "train_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Pipeline : {pipeline_path}")
print(f"Méta     : {FINAL_DIR / 'train_meta.json'}")

del X_train, X_train_s, X_train_p
gc.collect()

Pipeline : C:\Users\sh4rk\Documents\Zoidberg2.0\results\svm\final_model\svm_pipeline.joblib
Méta     : C:\Users\sh4rk\Documents\Zoidberg2.0\results\svm\final_model\train_meta.json


23057